### 欢迎来到第四周第三天——继续 LangGraph..

In [ ]:
from typing import Annotated
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langgraph.prebuilt import ToolNode, tools_condition
import requests
import os
from langchain_openai import ChatOpenAI
from typing import TypedDict


In [ ]:
# 我们最熟悉的第一步！顺便说一下，Crew 之前一直在帮我们做这个。
load_dotenv(override=True)

### 首先，我们去设置一下 LangSmith！

https://langsmith.com

### 接下来，这是 LangChain 社区中一个很有用的函数：

In [ ]:
from langchain_community.utilities import GoogleSerperAPIWrapper

serper = GoogleSerperAPIWrapper()
serper.run("What is the capital of France?")

### 现在来看一个 LangChain 包装类，它可以将函数转换成 Tool

In [ ]:
from langchain.agents import Tool

tool_search =Tool(
        name="search",
        func=serper.run,
        description="Useful for when you need more information from an online search"
    )



### 现在我们试试用 LangChain 的方式来调用这个工具

In [ ]:
tool_search.invoke("What is the capital of France?")

### 现在我们来自己写一个工具

选一个我们熟悉的

In [ ]:
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_url = "https://api.pushover.net/1/messages.json"

def push(text: str):
    """Send a push notification to the user"""
    requests.post(pushover_url, data = {"token": pushover_token, "user": pushover_user, "message": text})

In [ ]:
tool_push = Tool(
        name="send_push_notification",
        func=push,
        description="useful for when you want to send a push notification"
    )

tool_push.invoke("Hello, me")

### 回到昨天的 Graph

一个小变化——使用 TypedDict 而不是 BaseModel 来定义 State 对象

当我们实现工具时，总是需要对代码做 2 处修改：

1. 修改：在调用 OpenAI 时以 JSON 格式提供工具

2. 修改：处理返回结果：检查模型返回的 `finish_reason=="tool_calls"`，然后提取调用请求、运行函数、提供结果。

### 把它们整合到一起

In [ ]:
tools = [tool_search, tool_push]

In [ ]:
# 第 1 步：定义 State 对象
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
# 第 2 步：用这个 State 类启动 Graph Builder
graph_builder = StateGraph(State)

In [ ]:
# 这里不一样了：将工具绑定到 LLM 上
llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

In [ ]:
# 第 3 步：创建 Node


def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))

In [ ]:
# 第 4 步：创建 Edges（边）

# 条件边：根据 chatbot 的输出决定下一步走向
graph_builder.add_conditional_edges( "chatbot", tools_condition, "tools")

# 每次工具被调用后，返回 chatbot 来决定下一步
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

In [ ]:
# 第 5 步：编译 Graph
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

### 搞定！接下来，让我们做这个：

In [ ]:
def chat(user_input: str, history):
    result = graph.invoke({"messages": [{"role": "user", "content": user_input}]})
    return result["messages"][-1].content


gr.ChatInterface(chat, type="messages").launch()

## 好了，是时候加入 Memory（记忆）了！

### 但是等等！

我们已经有了这个 Graph 来维护状态并往状态里追加数据。

为什么它不能处理记忆呢？

### 这是理解 LangGraph 的一个关键点

> 一个 super-step（超级步骤）可以看作是对图中所有节点的一次迭代。并行运行的节点属于同一个 super-step，而串行运行的节点属于不同的 super-step。

Graph 的一个 "Super-Step" 代表了在 Agent 之间传递消息的一次调用。

在 LangGraph 惯用写法中，你为每个 super-step（每次交互）调用一次 `invoke` 来运行你的 graph。

**reducer 会在一个 super-step 内自动处理状态更新，但不会跨 super-step 处理。**

这就是 checkpointing（检查点）所要解决的问题。

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [ ]:
# 第 1 步和第 2 步
graph_builder = StateGraph(State)


# 第 3 步
llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    print(state)
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))

# 第 4 步
graph_builder.add_conditional_edges( "chatbot", tools_condition, "tools")
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

# 第 5 步：编译时传入 checkpointer 来持久化记忆
graph = graph_builder.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
config = {"configurable": {"thread_id": "1"}}

def chat(user_input: str, history):
    result = graph.invoke({"messages": [{"role": "user", "content": user_input}]}, config=config)
    return result["messages"][-1].content


gr.ChatInterface(chat, type="messages").launch()

In [ ]:
graph.get_state(config)

In [ ]:
# 最近的状态排在最前面
list(graph.get_state_history(config))

### LangGraph 为你提供了将状态回退到之前某个时间点、并创建分支的工具：

```
config = {"configurable": {"thread_id": "1", "checkpoint_id": ...}}
graph.invoke(None, config=config)
```

这使得你可以构建稳定的系统，能够从任意之前的检查点恢复并重新运行。

### 现在让我们把记忆存入 SQL

### 这就是 LangGraph 的强大之处。

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

db_path = "memory.db"
conn = sqlite3.connect(db_path, check_same_thread=False)
sql_memory = SqliteSaver(conn)

In [ ]:
# 第 1 步和第 2 步
graph_builder = StateGraph(State)


# 第 3 步
llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    print(state)
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))

# 第 4 步
graph_builder.add_conditional_edges( "chatbot", tools_condition, "tools")
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

# 第 5 步：使用 SQLite checkpointer 持久化记忆
graph = graph_builder.compile(checkpointer=sql_memory)
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
config = {"configurable": {"thread_id": "3"}}

def chat(user_input: str, history):
    result = graph.invoke({"messages": [{"role": "user", "content": user_input}]}, config=config)
    return result["messages"][-1].content


gr.ChatInterface(chat, type="messages").launch()